# Router Agent

In this notebook we will build a router agent, that depending on the user query will answer differently. We will be using LangGraph to create a graph of how the model should behave.

## Install Dependencies

In [ ]:
!sudo apt update
!sudo apt install -y python3-dev graphviz libgraphviz-dev pkg-config

In [ ]:
!pip install langgraph langchain pygraphviz langchain-openai langchain_mcp_adapters openmeteo_requests requests_cache retry_requests

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/ai-agents/ai-agents/open_weather.py -O open_weather.py

## Launch vLLM in the Background

Execute the cell below to create the vllm_serve.sh

In [ ]:
large_model = False
model_name = "Qwen3-30B-A3B" if large_model else "Qwen3-8B"

vllm_file = f"""#!/bin/bash

VLLM_USE_TRITON_FLASH_ATTN=0 \\
vllm serve Qwen/{model_name} \\
    --served-model-name {model_name} \\
    --api-key abc-123 \\
    --port 8000 \\
    --enable-auto-tool-choice \\
    --tool-call-parser hermes \\
    --trust-remote-code 2>&1 | tee vllm_serve.log
"""

with open('vllm_serve.sh', 'w', encoding='utf-8') as f:
    f.write(vllm_file)

Open a new terminal to execute the `vllm_serve.sh` file. This will serve an LLM locally.

In Jupyter, open a new terminal. `File > New > Terminal`, copy the content below and execute it.

```sh
bash vllm_serve.sh
```

Now, the LLM will be ready to be used once you see `Application startup complete.`

## Import Dependencies

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
import re
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

## Launch MCP Server

We are going to expose a MCP server that can provide current weather information.

In [ ]:
server_params = StdioServerParameters(command="python", args=['open_weather.py'])

## Create Agent State

To manage out agent state, we will create an object that can keep track of the messages. The `operator.add` makes sure that new messages are appended and not erased.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

## Create Router Agent

We will create the `Agent` class.

The `__init__` function initializes the `StateGraph` and defines the different nodes and edges of the graph. Note the conditional edge that depends on the `self.router_decision`. Once the graph is fully defined we can compile it. We will also initialize the model with a local Ollama endpoint.

The `self.router` takes the user query and uses a system prompt to get the LLM to provide the transition to the next state. Which will either provide an answer as a scientist or poet. 

`self.router_decision` simply gets the message state and returns the LLM response.

`self.play_writer` is called only when the agent identifies that the response should be in the form of a play. We use the system prompt to guide the model in the response. We also use the weather MCP tools to get current weather conditions in case a place is part of the query.

`scientist_writer` simply uses the system prompt to answer as a scientist.

In [ ]:
class Agent:
    def __init__(self):
        graph = StateGraph(AgentState)
        graph.add_node("router", self.router)
        graph.add_node("play_writer", self.play_writer)
        graph.add_node("scientist_writer", self.scientist_writer)
        graph.add_edge(START, "router")
        graph.add_edge("play_writer", END)
        graph.add_edge("scientist_writer", END)
        graph.add_conditional_edges(
            "router",
            self.router_decision,
            {"play": "play_writer", "scientist": "scientist_writer"}
        )
        self.graph = graph.compile()
        self.model = ChatOpenAI(model=model_name, base_url='http://localhost:8000/v1', api_key='abc-123')


    def router(self, state: AgentState):
        prompt = """You are a useful agent that takes the user query and based on the analysis of the query
        you return 'play' when a play writer is more applicable or 'scientist' when a scientific response is
        more applicable. Do not explain your reasoning, only return 'play' or 'scientist' always in lowercase.
        """
        user_query = state['messages']

        messages = [SystemMessage(content=prompt)] + user_query

        message = self.model.invoke(messages)
        return {'messages': [message]}

    def router_decision(self, state: AgentState):
        """Returns 'play' or 'scientist' """
        result = state['messages'][-1]
        content = result.content.lower()
        # Strip <think></think> tags if present
        content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
        return content

    async def play_writer(self, state: AgentState):
        user_query = state['messages'][-2]

        prompt = """You are a useful agent that writes plays in the style of William Shakespeare.
        Based on the user query provide a compelling yet short play.

        If you are provided with a place, pick a random location to get the current weather that
        should be part of the play. If you are provided with weather conditions use those as part of the poem.

        Sign the play with a reference to Shakespeare.
        """

        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                model_with_tools = self.model.bind_tools(tools)
                messages = [SystemMessage(content=prompt)] + [user_query]

                message = await model_with_tools.ainvoke(messages)
                if message.tool_calls:
                    for tool in message.tool_calls:
                        tool_call_result = await session.call_tool(tool['name'], arguments=tool['args'])
                        prompt = prompt + tool_call_result.content[-1].text
                        messages = [SystemMessage(content=prompt)] + [user_query]

                    message = await self.model.ainvoke(messages)

                return {'messages': [message]}

    def scientist_writer(self, state: AgentState):
        user_query = state['messages'][-2]
        prompt = """You are a useful agent that writes as if you were the scientist Stephen Hawking.
        Based on the user query provide a compelling yet short description.

        Sign the play with a reference to A Brief History of Time.
        """

        messages = [SystemMessage(content=prompt)] + [user_query]

        message = self.model.invoke(messages)
        return {'messages': [message]}

Let create an instance of our agent.

In [ ]:
writer_agent = Agent()

## Visualize Graph

We can now visualize what our graph looks like.

In [ ]:
from IPython.display import Image, display

display(Image(writer_agent.graph.get_graph().draw_png()))

## Use the Agent

Let's query the agent and try to get a story about the Mediterranean

In [ ]:
messages = [HumanMessage(content="Tell me a story about the Mediterranean.")]
response_mediterranean = await writer_agent.graph.ainvoke({"messages": messages})

Let's show the response of our agent

In [ ]:
response_mediterranean['messages'][-1].pretty_print()

You can also explore the full agent history

In [ ]:
for history in response_mediterranean['messages']:
    history.pretty_print()

Let's query the agent and try to get a scientific response

In [ ]:
messages = [HumanMessage(content="Tell me why the sky is blue")]
response_sky = writer_agent.graph.invoke({"messages": messages})

In [ ]:
response_sky['messages'][-1].pretty_print()

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT